# Advanced Python Problems with Solutions: Decorators and Descriptors

This notebook is a **problem-first, solution-driven** practice set for advanced Python metaprogramming with decorators and descriptors.

## Learning goals

By the end, you should be comfortable with:

- writing well-behaved decorators with `functools.wraps`
- building parameterized decorators
- preserving metadata and signatures
- stacking decorators and reasoning about application order
- decorating instance methods, class methods, static methods, generators, and async functions
- using class decorators
- understanding the descriptor protocol: `__get__`, `__set__`, `__delete__`, `__set_name__`
- distinguishing **data** and **non-data** descriptors
- understanding attribute lookup precedence
- storing per-instance descriptor state safely
- using `WeakKeyDictionary`
- building reusable validation descriptors
- implementing lazy/cached attributes
- combining decorators and descriptors in practical designs

## Best-practice rules used throughout

1. Use `functools.wraps` for function decorators.
2. Return the wrapped function's result unless intentionally changing the contract.
3. Prefer explicit, small decorators with a single responsibility.
4. Keep descriptor state per-instance, not on the descriptor itself.
5. Use `__set_name__` so descriptors discover their attribute names automatically.
6. Raise precise exceptions with helpful messages.
7. Preserve normal Python attribute access semantics whenever possible.
8. Add tests/assertions for behavior and edge cases.

In [1]:
from __future__ import annotations

import asyncio
import inspect
import time
from collections import Counter
from functools import wraps
from typing import Any, Callable, get_type_hints
from weakref import WeakKeyDictionary

## Warm-up Example 1 — A metadata-preserving debugger

Before the exercises, here is a compact reference implementation.

In [2]:
def debugger(fn):
    @wraps(fn)
    def wrapper(*args, **kwargs):
        print(f"CALL {fn.__qualname__} args={args} kwargs={kwargs}")
        result = fn(*args, **kwargs)
        print(f"RETURN {fn.__qualname__} -> {result!r}")
        return result
    return wrapper


@debugger
def add(a: int, b: int = 0) -> int:
    "Return a + b."
    return a + b


assert add(2, b=3) == 5
assert add.__name__ == "add"
assert add.__doc__ == "Return a + b."
inspect.signature(add)

CALL add args=(2,) kwargs={'b': 3}
RETURN add -> 5


<Signature (a: 'int', b: 'int' = 0) -> 'int'>

## Warm-up Example 2 — A reusable validated descriptor

In [3]:
class IntegerField:
    def __set_name__(self, owner, name):
        self.name = name

    def __get__(self, instance, owner=None):
        if instance is None:
            return self
        return instance.__dict__[self.name]

    def __set__(self, instance, value):
        if isinstance(value, bool) or not isinstance(value, int):
            raise TypeError(f"{self.name} must be an int")
        instance.__dict__[self.name] = value


class Point:
    x = IntegerField()
    y = IntegerField()

    def __init__(self, x, y):
        self.x = x
        self.y = y


p = Point(10, 20)
assert (p.x, p.y) == (10, 20)

try:
    p.x = 3.5
except TypeError as exc:
    print(exc)

x must be an int


## Problem 1 — Count calls without changing the wrapped function's interface

### Task

Write a decorator `count_calls` that:

- counts how many times the wrapped function is called,
- exposes the count as `wrapped.calls`,
- preserves metadata with `@wraps`,
- returns the original function's result.

Then test it on a recursive function and explain an important consequence of recursion.

### Solution

In [4]:
def count_calls(fn):
    @wraps(fn)
    def wrapper(*args, **kwargs):
        wrapper.calls += 1
        return fn(*args, **kwargs)

    wrapper.calls = 0
    return wrapper


@count_calls
def factorial(n: int) -> int:
    if n < 0:
        raise ValueError("n must be >= 0")
    if n < 2:
        return 1
    return n * factorial(n - 1)


assert factorial(5) == 120
assert factorial.calls == 5
assert factorial.__name__ == "factorial"

print("factorial(5) =", factorial(5))
print("total calls after calling twice =", factorial.calls)

# Because recursive calls use the decorated global name `factorial`,
# every recursive invocation also passes through the decorator.

factorial(5) = 120
total calls after calling twice = 10


## Problem 2 — Parameterized retry decorator

### Task

Implement `retry(attempts, exceptions)`.

Requirements:

- `attempts` must be at least 1.
- retry only the specified exception types.
- re-raise the final exception.
- preserve metadata.
- do not catch unrelated exceptions.

### Solution

In [5]:
def retry(attempts: int, exceptions=(Exception,)):
    if attempts < 1:
        raise ValueError("attempts must be >= 1")

    def decorate(fn):
        @wraps(fn)
        def wrapper(*args, **kwargs):
            for attempt in range(1, attempts + 1):
                try:
                    return fn(*args, **kwargs)
                except exceptions:
                    if attempt == attempts:
                        raise
        return wrapper

    return decorate


state = {"calls": 0}

@retry(attempts=3, exceptions=(ValueError,))
def flaky():
    state["calls"] += 1
    if state["calls"] < 3:
        raise ValueError("temporary failure")
    return "ok"


assert flaky() == "ok"
assert state["calls"] == 3

@retry(attempts=2, exceptions=(ValueError,))
def unrelated_failure():
    raise TypeError("do not retry this")


try:
    unrelated_failure()
except TypeError as exc:
    print("Correctly propagated:", exc)

Correctly propagated: do not retry this


## Problem 3 — Decorator application order

### Task

Predict the output before running it.

Create two decorators, `outer` and `inner`, that print messages before and after the wrapped call. Apply:

```python
@outer
@inner
def work():
    ...
```

Explain both:

1. decoration order,
2. call order.

### Solution

In [6]:
def outer(fn):
    @wraps(fn)
    def wrapper(*args, **kwargs):
        print("outer: before")
        result = fn(*args, **kwargs)
        print("outer: after")
        return result
    return wrapper


def inner(fn):
    @wraps(fn)
    def wrapper(*args, **kwargs):
        print("inner: before")
        result = fn(*args, **kwargs)
        print("inner: after")
        return result
    return wrapper


@outer
@inner
def work():
    print("work")


work()

# Equivalent decoration:
# work = outer(inner(work))
#
# Therefore call nesting is:
# outer before -> inner before -> work -> inner after -> outer after

outer: before
inner: before
work
inner: after
outer: after


## Problem 4 — Timing decorator with statistics

### Task

Build a decorator `timed` that tracks:

- number of calls,
- total elapsed time,
- minimum elapsed time,
- maximum elapsed time,
- average elapsed time.

Store the statistics on the wrapper as attributes. Use `time.perf_counter()`.

### Solution

In [7]:
def timed(fn):
    @wraps(fn)
    def wrapper(*args, **kwargs):
        start = time.perf_counter()
        try:
            return fn(*args, **kwargs)
        finally:
            elapsed = time.perf_counter() - start
            wrapper.calls += 1
            wrapper.total_time += elapsed
            wrapper.min_time = min(wrapper.min_time, elapsed)
            wrapper.max_time = max(wrapper.max_time, elapsed)

    wrapper.calls = 0
    wrapper.total_time = 0.0
    wrapper.min_time = float("inf")
    wrapper.max_time = 0.0

    def average_time():
        return wrapper.total_time / wrapper.calls if wrapper.calls else 0.0

    wrapper.average_time = average_time
    return wrapper


@timed
def small_job(n):
    return sum(i * i for i in range(n))


for n in (1_000, 2_000, 5_000):
    small_job(n)

print("calls:", small_job.calls)
print("total:", small_job.total_time)
print("min:", small_job.min_time)
print("max:", small_job.max_time)
print("avg:", small_job.average_time())

assert small_job.calls == 3
assert small_job.min_time <= small_job.max_time

calls: 3
total: 0.0011256001889705658
min: 8.639995940029621e-05
max: 0.0008673002012073994
avg: 0.0003752000629901886


## Problem 5 — Type-contract decorator using annotations

### Task

Create `enforce_types` that validates annotated positional/keyword arguments at runtime.

Scope this exercise deliberately:

- support ordinary concrete classes such as `int`, `str`, `float`,
- ignore parameters without annotations,
- ignore the return annotation,
- reject `bool` when an `int` is required.

Use `inspect.signature(...).bind(...)` instead of manually matching positional arguments.

### Solution

In [8]:
def enforce_types(fn):
    signature = inspect.signature(fn)
    type_hints = get_type_hints(fn)

    @wraps(fn)
    def wrapper(*args, **kwargs):
        bound = signature.bind(*args, **kwargs)
        bound.apply_defaults()

        for name, value in bound.arguments.items():
            annotation = type_hints.get(name, inspect.Parameter.empty)
            if annotation is inspect.Parameter.empty:
                continue

            if annotation is int:
                valid = isinstance(value, int) and not isinstance(value, bool)
            else:
                valid = isinstance(value, annotation)

            if not valid:
                raise TypeError(
                    f"{name} must be {annotation.__name__}; "
                    f"got {type(value).__name__}"
                )

        return fn(*args, **kwargs)

    return wrapper


@enforce_types
def repeat(text: str, count: int = 1):
    return text * count


assert repeat("ha", 3) == "hahaha"

for bad in [("ha", 2.5), ("ha", True)]:
    try:
        repeat(*bad)
    except TypeError as exc:
        print(exc)


count must be int; got float
count must be int; got bool


## Problem 6 — Memoization with unhashable argument protection

### Task

Implement a simple memoization decorator.

Requirements:

- cache results using `args` and sorted keyword pairs,
- expose the cache as `wrapper.cache`,
- if arguments are unhashable, bypass the cache instead of crashing.

### Solution

In [9]:
def memoize(fn):
    cache = {}

    @wraps(fn)
    def wrapper(*args, **kwargs):
        key = (args, tuple(sorted(kwargs.items())))
        try:
            hash(key)
        except TypeError:
            return fn(*args, **kwargs)

        if key not in cache:
            cache[key] = fn(*args, **kwargs)
        return cache[key]

    wrapper.cache = cache
    return wrapper


calls = Counter()

@memoize
def combine(a, b=0):
    calls["combine"] += 1
    return a + b


assert combine(10, b=5) == 15
assert combine(10, b=5) == 15
assert calls["combine"] == 1

@memoize
def list_length(values):
    return len(values)

# list is unhashable, so this works but is not cached.
assert list_length([1, 2, 3]) == 3
assert list_length([1, 2, 3]) == 3

print("combine cache:", combine.cache)

combine cache: {((10,), (('b', 5),)): 15}


## Problem 7 — Decorate an async function correctly

### Task

A normal synchronous wrapper is usually the wrong abstraction for an `async def`.

Implement `async_debugger` that:

- is itself an async wrapper,
- awaits the original coroutine,
- preserves metadata.

### Solution

In [10]:
def async_debugger(fn):
    if not inspect.iscoroutinefunction(fn):
        raise TypeError("async_debugger requires an async function")

    @wraps(fn)
    async def wrapper(*args, **kwargs):
        print(f"ASYNC CALL {fn.__qualname__}")
        result = await fn(*args, **kwargs)
        print(f"ASYNC RETURN {result!r}")
        return result

    return wrapper


@async_debugger
async def async_add(a, b):
    await asyncio.sleep(0)
    return a + b


result = await async_add(4, 7)
assert result == 11

ASYNC CALL async_add
ASYNC RETURN 11


## Problem 8 — Class decorator that registers subclasses

### Task

Create a class decorator `register(name)` that stores classes in a central registry.

Requirements:

- reject duplicate names,
- return the class unchanged,
- allow later construction through the registry.

### Solution

In [11]:
CLASS_REGISTRY = {}


def register(name: str):
    def decorate(cls):
        if name in CLASS_REGISTRY:
            raise KeyError(f"duplicate registration: {name!r}")
        CLASS_REGISTRY[name] = cls
        return cls
    return decorate


@register("json")
class JsonHandler:
    def handle(self, payload):
        return {"kind": "json", "payload": payload}


@register("text")
class TextHandler:
    def handle(self, payload):
        return f"TEXT:{payload}"


handler = CLASS_REGISTRY["json"]()
assert handler.handle(123) == {"kind": "json", "payload": 123}
print(CLASS_REGISTRY)

{'json': <class '__main__.JsonHandler'>, 'text': <class '__main__.TextHandler'>}


## Problem 9 — Descriptor supporting class-level access

### Task

Fix the classic descriptor bug:

```python
def __get__(self, instance, owner):
    return instance.__dict__[self.name]
```

This crashes when accessed as `Class.attribute`.

Implement a field descriptor that returns itself when `instance is None`.

### Solution

In [12]:
class StringField:
    def __set_name__(self, owner, name):
        self.name = name

    def __get__(self, instance, owner=None):
        if instance is None:
            return self
        return instance.__dict__[self.name]

    def __set__(self, instance, value):
        if not isinstance(value, str):
            raise TypeError(f"{self.name} must be a str")
        instance.__dict__[self.name] = value


class User:
    name = StringField()

    def __init__(self, name):
        self.name = name


u = User("Ada")
assert u.name == "Ada"
assert isinstance(User.name, StringField)

print("Instance access:", u.name)
print("Class access:", User.name)

Instance access: Ada
Class access: <__main__.StringField object at 0x000002E391E7EF90>


## Problem 10 — Data vs non-data descriptor precedence

### Task

Demonstrate the difference between:

- a **data descriptor** (`__get__` + `__set__` or `__delete__`),
- a **non-data descriptor** (`__get__` only).

Show that an instance dictionary entry can shadow a non-data descriptor but not a data descriptor.

### Solution

In [13]:
class NonData:
    def __get__(self, instance, owner=None):
        if instance is None:
            return self
        return "from NonData.__get__"


class Data:
    def __get__(self, instance, owner=None):
        if instance is None:
            return self
        return "from Data.__get__"

    def __set__(self, instance, value):
        instance.__dict__["_data_value"] = value


class Demo:
    non_data = NonData()
    data = Data()


d = Demo()

d.__dict__["non_data"] = "instance shadows non-data"
d.__dict__["data"] = "attempted instance shadow"

assert d.non_data == "instance shadows non-data"
assert d.data == "from Data.__get__"

print("non_data:", d.non_data)
print("data:", d.data)
print("instance dict:", d.__dict__)

non_data: instance shadows non-data
data: from Data.__get__
instance dict: {'non_data': 'instance shadows non-data', 'data': 'attempted instance shadow'}


## Problem 11 — Generic typed descriptor

### Task

Build `Typed` so that each descriptor instance validates a configured type.

Example target:

```python
class Product:
    name = Typed(str)
    price = Typed(float)
```

Add a special case so `bool` is not accepted for an `int` field.

### Solution

In [14]:
class Typed:
    def __init__(self, expected_type):
        self.expected_type = expected_type

    def __set_name__(self, owner, name):
        self.name = name

    def __get__(self, instance, owner=None):
        if instance is None:
            return self
        return instance.__dict__[self.name]

    def __set__(self, instance, value):
        if self.expected_type is int:
            valid = isinstance(value, int) and not isinstance(value, bool)
        else:
            valid = isinstance(value, self.expected_type)

        if not valid:
            raise TypeError(
                f"{self.name} must be {self.expected_type.__name__}; "
                f"got {type(value).__name__}"
            )
        instance.__dict__[self.name] = value


class Product:
    name = Typed(str)
    quantity = Typed(int)

    def __init__(self, name, quantity):
        self.name = name
        self.quantity = quantity


product = Product("keyboard", 3)
assert product.name == "keyboard"
assert product.quantity == 3

try:
    product.quantity = True
except TypeError as exc:
    print(exc)

quantity must be int; got bool


## Problem 12 — Range-validated descriptor with reusable constraints

### Task

Implement `Number` with configurable:

- numeric type,
- optional minimum,
- optional maximum.

Use `None` to mean unbounded. Produce attribute-specific error messages.

### Solution

In [15]:
class Number:
    def __init__(self, number_type=(int, float), *, minimum=None, maximum=None):
        self.number_type = number_type
        self.minimum = minimum
        self.maximum = maximum

    def __set_name__(self, owner, name):
        self.name = name

    def __get__(self, instance, owner=None):
        if instance is None:
            return self
        return instance.__dict__[self.name]

    def __set__(self, instance, value):
        if isinstance(value, bool) or not isinstance(value, self.number_type):
            raise TypeError(f"{self.name} must be numeric")

        if self.minimum is not None and value < self.minimum:
            raise ValueError(f"{self.name} must be >= {self.minimum}")

        if self.maximum is not None and value > self.maximum:
            raise ValueError(f"{self.name} must be <= {self.maximum}")

        instance.__dict__[self.name] = value


class Temperature:
    celsius = Number(minimum=-273.15)


t = Temperature()
t.celsius = 21.5
assert t.celsius == 21.5

try:
    t.celsius = -300
except ValueError as exc:
    print(exc)

celsius must be >= -273.15


## Problem 13 — Descriptor deletion protocol

### Task

Create a descriptor `RequiredField` that:

- stores a value,
- rejects deletion with `AttributeError`,
- raises `AttributeError` if read before assignment.

This introduces `__delete__`.

### Solution

In [16]:
class RequiredField:
    def __set_name__(self, owner, name):
        self.name = name

    def __get__(self, instance, owner=None):
        if instance is None:
            return self
        if self.name not in instance.__dict__:
            raise AttributeError(f"{self.name} has not been assigned")
        return instance.__dict__[self.name]

    def __set__(self, instance, value):
        instance.__dict__[self.name] = value

    def __delete__(self, instance):
        raise AttributeError(f"{self.name} cannot be deleted")


class Config:
    api_key = RequiredField()


cfg = Config()

try:
    print(cfg.api_key)
except AttributeError as exc:
    print(exc)

cfg.api_key = "secret"
assert cfg.api_key == "secret"

try:
    del cfg.api_key
except AttributeError as exc:
    print(exc)

api_key has not been assigned
api_key cannot be deleted


## Problem 14 — Per-instance descriptor storage with WeakKeyDictionary

### Task

Avoid writing descriptor-managed data into `instance.__dict__`.

Use `WeakKeyDictionary` so values disappear when instances are garbage-collected.

Important: descriptor state must still be per-instance.

### Solution

In [17]:
class WeakStorageField:
    def __init__(self):
        self._values = WeakKeyDictionary()

    def __set_name__(self, owner, name):
        self.name = name

    def __get__(self, instance, owner=None):
        if instance is None:
            return self
        try:
            return self._values[instance]
        except KeyError:
            raise AttributeError(f"{self.name} has not been assigned") from None

    def __set__(self, instance, value):
        self._values[instance] = value


class Session:
    token = WeakStorageField()


s1 = Session()
s2 = Session()

s1.token = "alpha"
s2.token = "beta"

assert s1.token == "alpha"
assert s2.token == "beta"
assert "token" not in s1.__dict__

print("s1.__dict__:", s1.__dict__)
print("s2.__dict__:", s2.__dict__)

s1.__dict__: {}
s2.__dict__: {}


## Problem 15 — Lazy non-data descriptor

### Task

Implement a `lazy` descriptor that computes a value once, then lets the instance dictionary shadow the descriptor.

This is a natural use-case for a **non-data descriptor** because it defines `__get__` but not `__set__`.

### Solution

In [18]:
class lazy:
    def __init__(self, fn):
        self.fn = fn
        wraps(fn)(self)

    def __set_name__(self, owner, name):
        self.name = name

    def __get__(self, instance, owner=None):
        if instance is None:
            return self

        value = self.fn(instance)
        instance.__dict__[self.name] = value
        return value


class Report:
    def __init__(self, values):
        self.values = values
        self.computations = 0

    @lazy
    def total(self):
        self.computations += 1
        return sum(self.values)


r = Report([10, 20, 30])

assert r.computations == 0
assert r.total == 60
assert r.computations == 1

# Second lookup resolves directly from r.__dict__,
# shadowing the non-data descriptor.
assert r.total == 60
assert r.computations == 1
assert r.__dict__["total"] == 60

print(r.__dict__)

{'values': [10, 20, 30], 'computations': 1, 'total': 60}


## Problem 16 — Cached data descriptor with explicit invalidation

### Task

Build a data descriptor `CachedProperty` with:

- lazy computation on first access,
- cached reads afterward,
- explicit invalidation via `del obj.attr`,
- read-only assignment behavior.

Unlike the previous exercise, this version defines `__set__`, so instance dictionary shadowing does not bypass it.

### Solution

In [19]:
class CachedProperty:
    def __init__(self, fn):
        self.fn = fn
        self.cache_name = None
        wraps(fn)(self)

    def __set_name__(self, owner, name):
        self.name = name
        self.cache_name = f"_{name}_cached"

    def __get__(self, instance, owner=None):
        if instance is None:
            return self

        if self.cache_name not in instance.__dict__:
            instance.__dict__[self.cache_name] = self.fn(instance)

        return instance.__dict__[self.cache_name]

    def __set__(self, instance, value):
        raise AttributeError(f"{self.name} is read-only")

    def __delete__(self, instance):
        instance.__dict__.pop(self.cache_name, None)


class Dataset:
    def __init__(self, values):
        self.values = values
        self.calls = 0

    @CachedProperty
    def mean(self):
        self.calls += 1
        return sum(self.values) / len(self.values)


data = Dataset([2, 4, 6])
assert data.mean == 4
assert data.mean == 4
assert data.calls == 1

del data.mean
assert data.mean == 4
assert data.calls == 2

try:
    data.mean = 99
except AttributeError as exc:
    print(exc)

mean is read-only


## Problem 17 — Reusable validation hierarchy

### Task

Refactor descriptors into a small hierarchy.

Create:

- `Validator` base descriptor,
- `NonEmptyString`,
- `PositiveInteger`.

The base class should own storage behavior, while subclasses only implement `validate`.

### Solution

In [20]:
class Validator:
    def __set_name__(self, owner, name):
        self.name = name

    def __get__(self, instance, owner=None):
        if instance is None:
            return self
        return instance.__dict__[self.name]

    def __set__(self, instance, value):
        self.validate(value)
        instance.__dict__[self.name] = value

    def validate(self, value):
        raise NotImplementedError


class NonEmptyString(Validator):
    def validate(self, value):
        if not isinstance(value, str):
            raise TypeError(f"{self.name} must be a str")
        if not value.strip():
            raise ValueError(f"{self.name} cannot be empty")


class PositiveInteger(Validator):
    def validate(self, value):
        if isinstance(value, bool) or not isinstance(value, int):
            raise TypeError(f"{self.name} must be an int")
        if value <= 0:
            raise ValueError(f"{self.name} must be > 0")


class InventoryItem:
    name = NonEmptyString()
    quantity = PositiveInteger()

    def __init__(self, name, quantity):
        self.name = name
        self.quantity = quantity


item = InventoryItem("SSD", 4)
assert (item.name, item.quantity) == ("SSD", 4)

for args in [("", 1), ("SSD", 0), ("SSD", True)]:
    try:
        InventoryItem(*args)
    except (TypeError, ValueError) as exc:
        print(type(exc).__name__, exc)

ValueError name cannot be empty
ValueError quantity must be > 0
TypeError quantity must be an int


## Problem 18 — Descriptor-generated audit log

### Task

Create an `AuditedField` that records every successful assignment.

Each audit entry should contain:

- attribute name,
- old value (or a sentinel representation),
- new value.

Store logs per instance under `_audit_log`.

### Solution

In [21]:
_MISSING = object()


class AuditedField:
    def __set_name__(self, owner, name):
        self.name = name

    def __get__(self, instance, owner=None):
        if instance is None:
            return self
        return instance.__dict__[self.name]

    def __set__(self, instance, value):
        old = instance.__dict__.get(self.name, _MISSING)
        instance.__dict__[self.name] = value

        log = instance.__dict__.setdefault("_audit_log", [])
        log.append({
            "field": self.name,
            "old": "<MISSING>" if old is _MISSING else old,
            "new": value,
        })


class Account:
    status = AuditedField()
    balance = AuditedField()

    def __init__(self, status, balance):
        self.status = status
        self.balance = balance


account = Account("open", 100)
account.balance = 150
account.status = "frozen"

for entry in account._audit_log:
    print(entry)

assert account._audit_log[-1] == {
    "field": "status",
    "old": "open",
    "new": "frozen",
}

{'field': 'status', 'old': '<MISSING>', 'new': 'open'}
{'field': 'balance', 'old': '<MISSING>', 'new': 100}
{'field': 'balance', 'old': 100, 'new': 150}
{'field': 'status', 'old': 'open', 'new': 'frozen'}


## Problem 19 — Decorator for generators

### Task

Write a decorator that counts how many values a generator function yields.

The decorator must remain lazy: calling the function should return a generator without consuming it immediately.

### Solution

In [22]:
def count_yields(fn):
    @wraps(fn)
    def wrapper(*args, **kwargs):
        wrapper.last_count = 0
        for value in fn(*args, **kwargs):
            wrapper.last_count += 1
            yield value

    wrapper.last_count = 0
    return wrapper


@count_yields
def countdown(n):
    while n > 0:
        yield n
        n -= 1


gen = countdown(3)
assert inspect.isgenerator(gen)
assert countdown.last_count == 0

assert list(gen) == [3, 2, 1]
assert countdown.last_count == 3

print("yield count:", countdown.last_count)

yield count: 3


## Problem 20 — Decorator that works for both sync and async functions

### Task

Create `trace` that accepts either a normal function or an async function and returns the appropriate kind of wrapper.

Use `inspect.iscoroutinefunction`.

### Solution

In [23]:
def trace(fn):
    if inspect.iscoroutinefunction(fn):
        @wraps(fn)
        async def async_wrapper(*args, **kwargs):
            print(f"TRACE async {fn.__qualname__}")
            return await fn(*args, **kwargs)
        return async_wrapper

    @wraps(fn)
    def sync_wrapper(*args, **kwargs):
        print(f"TRACE sync {fn.__qualname__}")
        return fn(*args, **kwargs)

    return sync_wrapper


@trace
def multiply(a, b):
    return a * b


@trace
async def async_multiply(a, b):
    await asyncio.sleep(0)
    return a * b


assert multiply(6, 7) == 42
assert await async_multiply(6, 7) == 42
assert not inspect.iscoroutinefunction(multiply)
assert inspect.iscoroutinefunction(async_multiply)

TRACE sync multiply
TRACE async async_multiply


## Problem 21 — Decorating methods: ordering with @classmethod and @staticmethod

### Task

Explore decorator order for descriptors built into Python.

Create a `log_call` decorator and apply it correctly to:

- an instance method,
- a `@classmethod`,
- a `@staticmethod`.

A robust convention is to put `@log_call` **closest to the function** and `@classmethod` / `@staticmethod` above it.

### Solution

In [24]:
def log_call(fn):
    @wraps(fn)
    def wrapper(*args, **kwargs):
        print(f"calling {fn.__qualname__}")
        return fn(*args, **kwargs)
    return wrapper


class MathTools:
    @log_call
    def instance_double(self, x):
        return x * 2

    @classmethod
    @log_call
    def class_name(cls):
        return cls.__name__

    @staticmethod
    @log_call
    def square(x):
        return x * x


tools = MathTools()

assert tools.instance_double(5) == 10
assert MathTools.class_name() == "MathTools"
assert MathTools.square(9) == 81

calling MathTools.instance_double
calling MathTools.class_name
calling MathTools.square


## Problem 22 — Decorating every public method in a class

### Task

Write a class decorator `debug_public_methods` that automatically decorates every public method stored directly in the class namespace.

For this exercise:

- skip names beginning with `_`,
- decorate ordinary functions,
- preserve all other class attributes unchanged.

### Solution

In [25]:
def debug_public_methods(cls):
    for name, value in list(vars(cls).items()):
        if name.startswith("_"):
            continue
        if inspect.isfunction(value):
            setattr(cls, name, debugger(value))
    return cls


@debug_public_methods
class Calculator:
    version = 1

    def add(self, a, b):
        return a + b

    def multiply(self, a, b):
        return a * b

    def _helper(self):
        return "private"


calc = Calculator()
assert calc.add(2, 3) == 5
assert calc.multiply(4, 5) == 20
assert calc._helper() == "private"
assert Calculator.version == 1

CALL Calculator.add args=(<__main__.Calculator object at 0x000002E391F75160>, 2, 3) kwargs={}
RETURN Calculator.add -> 5
CALL Calculator.multiply args=(<__main__.Calculator object at 0x000002E391F75160>, 4, 5) kwargs={}
RETURN Calculator.multiply -> 20


## Problem 23 — Descriptor inheritance and owner reporting

### Task

Implement a descriptor that reports the class through which it was accessed.

Use the `owner` argument of `__get__` and demonstrate access through both a base class and subclass.

### Solution

In [26]:
class OwnerAware:
    def __set_name__(self, owner, name):
        self.name = name
        self.declared_on = owner

    def __get__(self, instance, owner=None):
        owner = owner or type(instance)
        return {
            "attribute": self.name,
            "declared_on": self.declared_on.__name__,
            "accessed_through": owner.__name__,
            "instance_access": instance is not None,
        }


class Base:
    info = OwnerAware()


class Child(Base):
    pass


print(Base.info)
print(Child.info)
print(Child().info)

assert Base.info["declared_on"] == "Base"
assert Child.info["accessed_through"] == "Child"
assert Child().info["instance_access"] is True

{'attribute': 'info', 'declared_on': 'Base', 'accessed_through': 'Base', 'instance_access': False}
{'attribute': 'info', 'declared_on': 'Base', 'accessed_through': 'Child', 'instance_access': False}
{'attribute': 'info', 'declared_on': 'Base', 'accessed_through': 'Child', 'instance_access': True}


## Problem 24 — Mini framework: descriptors + decorator + validation

### Task

Combine both topics.

Build:

1. a `ValidatedField` descriptor accepting a validator function,
2. a `validate_call` decorator that verifies a method leaves the object in a valid state,
3. a `BankAccount` class with validated `owner` and `balance`.

Rules:

- owner must be a non-empty string,
- balance must be numeric and non-negative,
- withdrawals may not overdraw the account.

### Solution

In [27]:
class ValidatedField:
    def __init__(self, validator: Callable[[Any], None]):
        self.validator = validator

    def __set_name__(self, owner, name):
        self.name = name

    def __get__(self, instance, owner=None):
        if instance is None:
            return self
        return instance.__dict__[self.name]

    def __set__(self, instance, value):
        self.validator(value)
        instance.__dict__[self.name] = value


def require_owner(value):
    if not isinstance(value, str):
        raise TypeError("owner must be a string")
    if not value.strip():
        raise ValueError("owner cannot be empty")


def require_balance(value):
    if isinstance(value, bool) or not isinstance(value, (int, float)):
        raise TypeError("balance must be numeric")
    if value < 0:
        raise ValueError("balance cannot be negative")


def validate_call(fn):
    @wraps(fn)
    def wrapper(self, *args, **kwargs):
        result = fn(self, *args, **kwargs)
        require_owner(self.owner)
        require_balance(self.balance)
        return result
    return wrapper


class BankAccount:
    owner = ValidatedField(require_owner)
    balance = ValidatedField(require_balance)

    def __init__(self, owner, balance=0):
        self.owner = owner
        self.balance = balance

    @validate_call
    def deposit(self, amount):
        if amount <= 0:
            raise ValueError("deposit must be > 0")
        self.balance += amount
        return self.balance

    @validate_call
    def withdraw(self, amount):
        if amount <= 0:
            raise ValueError("withdrawal must be > 0")
        if amount > self.balance:
            raise ValueError("insufficient funds")
        self.balance -= amount
        return self.balance


acct = BankAccount("Ada", 100)
assert acct.deposit(50) == 150
assert acct.withdraw(40) == 110

try:
    acct.withdraw(1_000)
except ValueError as exc:
    print(exc)

assert acct.balance == 110

insufficient funds


## Problem 25 — Challenge: transactional method decorator with rollback

### Task

Implement `rollback_on_error(*attribute_names)`.

Before calling the method:

- snapshot selected attributes.

If the method raises:

- restore the original values,
- re-raise the exception.

This is useful for seeing how decorators can add object-level transactional behavior.

### Solution

In [28]:
def rollback_on_error(*attribute_names):
    def decorate(fn):
        @wraps(fn)
        def wrapper(self, *args, **kwargs):
            snapshot = {
                name: getattr(self, name)
                for name in attribute_names
            }

            try:
                return fn(self, *args, **kwargs)
            except Exception:
                for name, value in snapshot.items():
                    setattr(self, name, value)
                raise

        return wrapper
    return decorate


class Wallet:
    balance = Number(minimum=0)

    def __init__(self, balance):
        self.balance = balance

    @rollback_on_error("balance")
    def risky_purchase(self, amount):
        self.balance -= amount

        # Simulate a later failure after state mutation.
        if amount > 50:
            raise RuntimeError("payment processor failed")

        return self.balance


wallet = Wallet(100)
assert wallet.risky_purchase(25) == 75

try:
    wallet.risky_purchase(60)
except RuntimeError as exc:
    print(exc)

# Rollback restored the balance from 15 to 75.
assert wallet.balance == 75
print("balance after rollback:", wallet.balance)

payment processor failed
balance after rollback: 75


# Additional Concept Checks

These are short, advanced prompts you can answer without running code first.

1. Why should a descriptor generally return itself when `instance is None`?
2. Why can a non-data descriptor be shadowed by `instance.__dict__`?
3. Why does adding even a simple `__set__` method change lookup precedence?
4. Why is storing `self.value` inside a descriptor usually wrong for normal per-instance fields?
5. When is `WeakKeyDictionary` useful for descriptor storage?
6. Why does `@wraps` matter beyond making `__name__` look nice?
7. Why must an async decorator `await` the wrapped coroutine?
8. What changes when a decorator wraps a generator function with `yield from` / iteration?
9. Why can decorator order change behavior?
10. In what situations is a descriptor preferable to repeated `@property` definitions?

## Concept Check Answers

1. **Class-level access:** `Class.attr` calls `descriptor.__get__(None, Class)`. Returning the descriptor itself supports introspection and configuration.
2. **Lookup precedence:** instance attributes are checked before non-data descriptors.
3. **Data descriptor priority:** defining `__set__` or `__delete__` makes the descriptor a data descriptor, which has higher lookup priority than the instance dictionary.
4. **Shared descriptor object:** the descriptor lives on the class and is shared by all instances. `self.value` would therefore be shared state.
5. **External per-instance storage:** `WeakKeyDictionary` is helpful when the descriptor should manage storage without writing into the instance dictionary and without unnecessarily keeping instances alive.
6. **Introspection/tooling:** `@wraps` also sets `__wrapped__`, enabling tools such as `inspect.signature` to discover the original callable.
7. **Coroutine semantics:** calling an async function produces a coroutine object; the wrapper must await it to obtain the result and preserve async behavior.
8. **Laziness:** a generator decorator should normally avoid eagerly converting the generator to a list, otherwise it changes memory use and evaluation timing.
9. **Composition:** `@a @b` means `a(b(fn))`, so the outer wrapper sees the behavior produced by the inner wrapper.
10. **DRY cross-cutting field behavior:** descriptors are useful when many attributes/classes need the same access, validation, transformation, logging, or lazy-loading behavior.

# Final Capstone — Build It Yourself

Implement a small model system with these features:

- `Field` base descriptor.
- `StringField(min_length=..., max_length=...)`.
- `IntegerField(minimum=..., maximum=...)`.
- values stored in `instance.__dict__`.
- automatic field discovery from the class dictionary.
- `@audit_methods` class decorator that wraps all public instance methods.
- a `Model.as_dict()` method returning field values.
- a `Model.update(**changes)` method that uses the descriptors for validation.
- rollback if any update fails.

Suggested model:

```python
class User(Model):
    name = StringField(min_length=1, max_length=50)
    age = IntegerField(minimum=0, maximum=130)
```

Try implementing it before revealing the solution below.

## Capstone Solution

In [29]:
class Field:
    def __set_name__(self, owner, name):
        self.name = name

    def __get__(self, instance, owner=None):
        if instance is None:
            return self
        if self.name not in instance.__dict__:
            raise AttributeError(f"{self.name} has not been assigned")
        return instance.__dict__[self.name]

    def __set__(self, instance, value):
        self.validate(value)
        instance.__dict__[self.name] = value

    def validate(self, value):
        raise NotImplementedError


class StringField(Field):
    def __init__(self, *, min_length=0, max_length=None):
        self.min_length = min_length
        self.max_length = max_length

    def validate(self, value):
        if not isinstance(value, str):
            raise TypeError(f"{self.name} must be a string")
        if len(value) < self.min_length:
            raise ValueError(
                f"{self.name} must have length >= {self.min_length}"
            )
        if self.max_length is not None and len(value) > self.max_length:
            raise ValueError(
                f"{self.name} must have length <= {self.max_length}"
            )


class IntegerField(Field):
    def __init__(self, *, minimum=None, maximum=None):
        self.minimum = minimum
        self.maximum = maximum

    def validate(self, value):
        if isinstance(value, bool) or not isinstance(value, int):
            raise TypeError(f"{self.name} must be an int")
        if self.minimum is not None and value < self.minimum:
            raise ValueError(f"{self.name} must be >= {self.minimum}")
        if self.maximum is not None and value > self.maximum:
            raise ValueError(f"{self.name} must be <= {self.maximum}")


def audit_methods(cls):
    for name, value in list(vars(cls).items()):
        if name.startswith("_") or not inspect.isfunction(value):
            continue

        def make_wrapper(fn):
            @wraps(fn)
            def wrapper(self, *args, **kwargs):
                self.__dict__.setdefault("_method_audit", []).append(
                    {"method": fn.__name__, "args": args, "kwargs": dict(kwargs)}
                )
                return fn(self, *args, **kwargs)
            return wrapper

        setattr(cls, name, make_wrapper(value))

    return cls


class Model:
    @classmethod
    def fields(cls):
        discovered = {}

        # Walk bases first so subclasses may override fields.
        for base in reversed(cls.__mro__):
            for name, value in vars(base).items():
                if isinstance(value, Field):
                    discovered[name] = value

        return discovered

    def as_dict(self):
        return {
            name: getattr(self, name)
            for name in self.fields()
        }

    def update(self, **changes):
        unknown = set(changes) - set(self.fields())
        if unknown:
            raise AttributeError(
                f"unknown field(s): {', '.join(sorted(unknown))}"
            )

        snapshot = self.as_dict()

        try:
            for name, value in changes.items():
                setattr(self, name, value)
        except Exception:
            for name, old_value in snapshot.items():
                setattr(self, name, old_value)
            raise

        return self


@audit_methods
class User(Model):
    name = StringField(min_length=1, max_length=50)
    age = IntegerField(minimum=0, maximum=130)

    def __init__(self, name, age):
        self.name = name
        self.age = age

    def birthday(self):
        self.age += 1
        return self.age


user = User("Ada", 36)

assert user.as_dict() == {"name": "Ada", "age": 36}
assert user.birthday() == 37
assert user._method_audit[-1]["method"] == "birthday"

user.update(name="Grace", age=40)
assert user.as_dict() == {"name": "Grace", "age": 40}

before = user.as_dict()

try:
    user.update(name="Valid Name", age=999)
except ValueError as exc:
    print("Update rejected:", exc)

# Transactional rollback:
assert user.as_dict() == before

try:
    user.update(unknown=123)
except AttributeError as exc:
    print("Unknown field rejected:", exc)

print("Final model:", user.as_dict())
print("Method audit:", user._method_audit)

Update rejected: age must be <= 130
Unknown field rejected: unknown field(s): unknown
Final model: {'name': 'Grace', 'age': 40}
Method audit: [{'method': 'birthday', 'args': (), 'kwargs': {}}]


# Summary Cheat Sheet

## Decorator skeleton

```python
from functools import wraps

def decorator(fn):
    @wraps(fn)
    def wrapper(*args, **kwargs):
        # before
        result = fn(*args, **kwargs)
        # after
        return result
    return wrapper
```

## Parameterized decorator skeleton

```python
def decorator_factory(option):
    def decorator(fn):
        @wraps(fn)
        def wrapper(*args, **kwargs):
            ...
            return fn(*args, **kwargs)
        return wrapper
    return decorator
```

## Descriptor skeleton

```python
class Field:
    def __set_name__(self, owner, name):
        self.name = name

    def __get__(self, instance, owner=None):
        if instance is None:
            return self
        return instance.__dict__[self.name]

    def __set__(self, instance, value):
        instance.__dict__[self.name] = value
```

## Attribute lookup idea

A useful simplified mental model for `obj.attr` is:

1. data descriptor on the class / MRO,
2. `obj.__dict__`,
3. non-data descriptor or ordinary class attribute,
4. `__getattr__` fallback, if defined.

The exact mechanics are implemented by Python's attribute access machinery, especially `object.__getattribute__`.

## Key design question

Use a **decorator** when you want to modify callable/class behavior at definition time.

Use a **descriptor** when you want reusable control over attribute access.